
# KSC StationXML builder (merged and cleaned)

This notebook merges the strongest parts of the uploaded student examples into one clearer version.

## What this version fixes

- uses the **ObsPy NRL** correctly for the **Nanometrics Trillium Compact 120 + Centaur** response
- uses a **sample rate of 100 Hz**, consistent with the NRL datalogger path used below
- uses **BHZ/BHN/BHE** channel codes, which match a broadband sensor sampled at 100 Hz
- assigns standard component orientations:
  - **Z**: azimuth 0, dip -90
  - **N**: azimuth 0, dip 0
  - **E**: azimuth 90, dip 0
- reads either the Excel file or a CSV fallback
- adds clear comments so you can adapt it later for other KSC deployments

## Assumptions

This version assumes the seven USF stations all used the same Nanometrics hardware described in the homework:

- **Digitizer:** Nanometrics Centaur
- **Sensor:** Nanometrics Trillium Compact Post-Hole 120 s

If any station had different hardware, you should branch on that row and attach a different response for that station.


In [ ]:

from pathlib import Path
import pandas as pd
from obspy import UTCDateTime
from obspy.clients.nrl import NRL
from obspy.core.inventory import Inventory, Network, Station, Channel, Site



## 1. Read the station table

Put one of these files in the same folder as this notebook:

- `Summary_Seismic_Station_List.xlsx`
- `Summary_Seismic_Station_List.csv`

The code below prefers Excel, but falls back to CSV.


In [ ]:

excel_path = Path("Summary_Seismic_Station_List.xlsx")
csv_path = Path("Summary_Seismic_Station_List.csv")

if excel_path.exists():
    df = pd.read_excel(excel_path)
    input_path = excel_path
elif csv_path.exists():
    df = pd.read_csv(csv_path)
    input_path = csv_path
else:
    raise FileNotFoundError(
        "Could not find Summary_Seismic_Station_List.xlsx or Summary_Seismic_Station_List.csv"
    )

print(f"Loaded station table from: {input_path}")
df.head()



## 2. Inspect and normalise the columns

Different students used slightly different assumptions about the station table.
This helper makes the code more robust to column name variations such as:

- `Site Name` vs `site name`
- `lat` vs `latitude`
- `lon` vs `longitude`


In [ ]:

def find_column(columns, candidates):
    """Return the first matching column name, ignoring case and extra spaces."""
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        key = candidate.strip().lower()
        if key in lookup:
            return lookup[key]
    raise KeyError(f"Could not find any of these columns: {candidates}")

sta_col = find_column(df.columns, ["Site Name", "station", "station code", "sta", "site"])
lat_col = find_column(df.columns, ["lat", "latitude"])
lon_col = find_column(df.columns, ["lon", "longitude"])

print("Station column:", sta_col)
print("Latitude column:", lat_col)
print("Longitude column:", lon_col)



## 3. Load the instrument response from the ObsPy NRL

This is the cleanest part of several student solutions.

### Important correction
Some student versions used:

- NRL datalogger key ending in `"100"`
- but a channel sample rate of `500.0`

Those two choices are inconsistent.

Because the NRL path below explicitly selects the **100 sps** Centaur configuration,
this merged version uses **100.0 Hz** and therefore **BHZ/BHN/BHE** channel codes.


In [ ]:

nrl = NRL()

sensor_keys = [
    "Nanometrics",
    "Trillium Compact 120 (Vault, Posthole, OBS)",
    "754 V/m/s",
]

datalogger_keys = [
    "Nanometrics",
    "Centaur",
    "40 Vpp (1)",
    "Off",
    "Linear phase",
    "100",
]

response = nrl.get_response(
    sensor_keys=sensor_keys,
    datalogger_keys=datalogger_keys,
)

print(response)



## 4. Define network- and channel-level metadata

### Notes

- `NETWORK_CODE = "1R"` is kept from the homework starter.
- `START_DATE` is set to **2026-01-07**, matching the beginning of the deployment window.
- `BHZ/BHN/BHE` is used because we are treating this as a broadband 100 Hz installation.
- `ELEV_M = 0.0` is only a placeholder. Replace it with actual station elevations if you have them.
- `DEPTH_M = 0.75` follows the burial-depth assumption used in the student notebooks.


In [ ]:

NETWORK_CODE = "1R"
NETWORK_DESCRIPTION = "Kennedy Space Center USF seismic deployment"
START_DATE = UTCDateTime(2026, 1, 7)

LOCATION_CODE = ""
SAMPLE_RATE_HZ = 100.0
CHANNEL_CODES = ["BHZ", "BHN", "BHE"]
ELEV_M = 0.0
DEPTH_M = 0.75
SITE_DESCRIPTION = "Kennedy Space Center"

# Standard channel orientations in SEED/StationXML convention.
# Dip is degrees downward from horizontal:
#   Z up = -90
#   N horizontal = 0
#   E horizontal = 0
ORIENTATION = {
    "BHZ": {"azimuth": 0.0, "dip": -90.0},
    "BHN": {"azimuth": 0.0, "dip": 0.0},
    "BHE": {"azimuth": 90.0, "dip": 0.0},
}



## 5. Build one `Station` object per row

This is where we combine the best student ideas:

- read coordinates from the table
- attach the same response to every USF seismic channel
- include azimuth and dip
- create three channels per station


In [ ]:

def make_channel(code, latitude, longitude, elevation_m, depth_m, sample_rate_hz, response):
    """Create one ObsPy Channel with correct orientation metadata."""
    orient = ORIENTATION[code]
    return Channel(
        code=code,
        location_code=LOCATION_CODE,
        latitude=float(latitude),
        longitude=float(longitude),
        elevation=float(elevation_m),
        depth=float(depth_m),
        azimuth=float(orient["azimuth"]),
        dip=float(orient["dip"]),
        sample_rate=float(sample_rate_hz),
        response=response,
    )


stations = []

for _, row in df.iterrows():
    sta_code = str(row[sta_col]).strip()
    latitude = float(row[lat_col])
    longitude = float(row[lon_col])

    channels = [
        make_channel(
            code=chan_code,
            latitude=latitude,
            longitude=longitude,
            elevation_m=ELEV_M,
            depth_m=DEPTH_M,
            sample_rate_hz=SAMPLE_RATE_HZ,
            response=response,
        )
        for chan_code in CHANNEL_CODES
    ]

    station = Station(
        code=sta_code,
        latitude=latitude,
        longitude=longitude,
        elevation=ELEV_M,
        creation_date=START_DATE,
        site=Site(name=f"{SITE_DESCRIPTION} {sta_code}"),
        channels=channels,
    )

    stations.append(station)

print(f"Built {len(stations)} Station objects")



## 6. Wrap the stations in a `Network` and `Inventory`


In [ ]:

network = Network(
    code=NETWORK_CODE,
    description=NETWORK_DESCRIPTION,
    start_date=START_DATE,
    stations=stations,
)

inventory = Inventory(
    networks=[network],
    source="Merged and cleaned from student KSC StationXML homework examples",
)

print(inventory)
inventory



## 7. Quick sanity checks

These are useful before writing StationXML.


In [ ]:

# Basic inventory summary
for net in inventory:
    print(f"Network {net.code}: {len(net.stations)} stations")
    for sta in net:
        print(f"  {sta.code}: {len(sta.channels)} channels")

# Confirm that every channel has a response attached
missing = []
for net in inventory:
    for sta in net:
        for cha in sta:
            if cha.response is None:
                missing.append((net.code, sta.code, cha.code))

print("Missing responses:", missing if missing else "None")



## 8. Optional plots

Uncomment these if you want a quick visual check.


In [ ]:

# inventory.plot(projection="local", level="channel")
# inventory.plot_response(min_freq=0.001, output="VEL", station=stations[0].code, channel="BHZ")



## 9. Write the StationXML file


In [ ]:

out_xml = Path("ksc_1R_2026_usf_seismic.xml")
inventory.write(str(out_xml), format="STATIONXML", validate=True)
print(f"Wrote {out_xml.resolve()}")



## Final remarks

### Why this merged version is better

It keeps the strongest parts of the student attempts, but corrects the main technical issues:

1. **Sample rate now matches the NRL path**.
   - Student examples often used the Centaur `100` configuration but set the channel sample rate to `500.0`.
   - That mismatch is removed here.

2. **Channel codes now match the sample rate and broadband sensor choice**.
   - This version uses **BHZ/BHN/BHE**.

3. **Orientation metadata is explicit**.
   - Several versions omitted `azimuth` and `dip`, or used a non-standard Z dip.

4. **The code is easier to adapt**.
   - You can now swap in real elevation values, different responses, or a different network code without rewriting the whole notebook.

### Things you may still want to improve later

- read true station elevation from the station file instead of using `0.0`
- attach station start/end dates individually if the stations were installed on different days
- add equipment metadata if you want a richer StationXML file
- add Marshall stations separately if you find the Guralp and Silicon Audio responses
